In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data = pd.read_csv("plagiarism_corpus.csv")

print("Dataset Shape:", data.shape)
print("\nFirst 5 Documents:")
print(data.head())

# CHECK MISSING VALUES AND DUPLICATES

print("\nMissing Values:")
print(data.isnull().sum())

print("\nDuplicate Documents:")
print(data["text"].duplicated().sum())

# Remove missing documents
data = data.dropna(subset=["text"])

# Remove exact duplicate documents
data = data.drop_duplicates(subset=["text"])

# TEXT CLEANING

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove punctuation and numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


data["cleaned_text"] = data["text"].apply(clean_text)

# DISPLAY ORIGINAL VS CLEANED TEXT

print("\nOriginal vs Cleaned Text:")
print("--------------------------------")

print(
    data[["text", "cleaned_text"]].head()
)

# TF-IDF VECTORIZATION

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    data["cleaned_text"]
)

print("\nTF-IDF Matrix Shape:")
print(tfidf_matrix.shape)

# CALCULATE COSINE SIMILARITY

similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print("\nSimilarity Matrix:")
print(similarity_matrix)

# CREATE SIMILARITY DATAFRAME

document_names = data["document_id"].tolist()

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=document_names,
    columns=document_names
)

print("\nSimilarity Matrix:")
print(similarity_df.round(2))

# SET PLAGIARISM THRESHOLD

threshold = 0.70

# FIND DOCUMENT PAIRS

results = []

for i in range(len(document_names)):

    for j in range(i + 1, len(document_names)):

        score = similarity_matrix[i][j]

        percentage = score * 100

        if score >= threshold:
            status = "Possible Plagiarism"
        else:
            status = "Low Similarity"

        results.append([
            document_names[i],
            document_names[j],
            round(percentage, 2),
            status
        ])

# CREATE REPORT

report = pd.DataFrame(
    results,
    columns=[
        "Document 1",
        "Document 2",
        "Similarity (%)",
        "Status"
    ]
)

# 11. RANK BY SIMILARITY

report = report.sort_values(
    by="Similarity (%)",
    ascending=False
)


print("\n============================================")
print("PLAGIARISM SIMILARITY REPORT")
print("============================================")

print(report.to_string(index=False))

# 12. DISPLAY MOST SUSPICIOUS PAIRS

print("\n============================================")
print("SUSPICIOUS DOCUMENT PAIRS")
print("============================================")

suspicious = report[
    report["Similarity (%)"] >= threshold * 100
]

if len(suspicious) > 0:

    print(
        suspicious.to_string(index=False)
    )

else:

    print("No potentially copied documents found.")

# 13. EXPORT REPORT

report.to_csv(
    "plagiarism_similarity_report.csv",
    index=False
)

similarity_df.to_csv(
    "similarity_matrix.csv"
)

print("\nReports saved successfully!")

Dataset Shape: (15, 2)

First 5 Documents:
    document_id                                               text
0  Assignment_A  Machine learning is a branch of artificial int...
1  Assignment_B  Machine learning is a field of artificial inte...
2  Assignment_C  Natural language processing is a branch of art...
3  Assignment_D  Machine learning algorithms learn from histori...
4  Assignment_E  Natural language processing allows computers t...

Missing Values:
document_id    0
text           0
dtype: int64

Duplicate Documents:
0

Original vs Cleaned Text:
--------------------------------
                                                text  \
0  Machine learning is a branch of artificial int...   
1  Machine learning is a field of artificial inte...   
2  Natural language processing is a branch of art...   
3  Machine learning algorithms learn from histori...   
4  Natural language processing allows computers t...   

                                        cleaned_text  
0  machine lear